In [6]:
# import pandas as pd
# import glob
# import os
# import io
# # 指定CSV文件所在的文件夹路径
# csv_folder_path = r"/mnt/c/csv/"
# # 指定Parquet文件将要保存的文件夹路径
# parquet_folder_path = r"/mnt/c/parquet/"

# # 使用glob模块获取所有CSV文件的路径
# csv_files = glob.glob(os.path.join(csv_folder_path, '*.csv'))

# # 遍历所有CSV文件
# for csv_file in csv_files:
#     # 读取CSV文件
#     df = pd.read_csv(csv_file)
#     df = df.astype(str).replace({'nan': None})
#     file_name = os.path.basename(csv_file)
#     parquet_file = os.path.join(parquet_folder_path, file_name.replace('.csv', '.parquet'))
    
#     # 写入Parquet文件
#     df.to_parquet(parquet_file, engine='pyarrow')
#     print(f"Converted {file_name} to Parquet format.")

# print("All files have been converted to Parquet format.")


In [1]:
from pyspark.sql import SparkSession

# 强制停止旧 Session，释放 JVM 句柄
if 'spark' in locals():
    spark.stop()

spark = (SparkSession.builder
    .appName("Phil_IP_Bridge_POC")
    .master("local[2]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    
    # --- 使用物理 IP 避开 localhost 陷阱 ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://172.22.19.65:9000") 
    
    # Iceberg Catalog 配置
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://bc2-standardised-restricted-ide/warehouse")
    
    # MinIO 认证
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    
    # --- 核心优化：禁止死循环重试 ---
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "5000")
    .config("spark.hadoop.fs.s3a.retry.limit", "1")  # 只重试一次
    .config("spark.hadoop.fs.s3a.retry.interval", "1s")
    .getOrCreate())

print("🚀 IP 桥接 Session 已成功初始化！")

# --- 验证 ---
try:
    spark.sql("SHOW DATABASES IN local").show()
    print("✅ 数据库列表拉取成功！")
except Exception as e:
    print(f"❌ 还是不行，请尝试在浏览器访问 http://172.22.19.65:9000 查看是否有 XML 输出。")
    print(f"具体报错：{e}")

your 131072x1 screen size is bogus. expect trouble
26/04/01 10:11:34 WARN Utils: Your hostname, DESKTOP-CDCLH86 resolves to a loopback address: 127.0.1.1; using 172.22.19.65 instead (on interface eth0)
26/04/01 10:11:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/phil/ldp/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/phil/.ivy2/cache
The jars for the packages stored in: /home/phil/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-24189a61-5e34-4439-b6fc-27893c81edb9;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 185ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.

🚀 IP 桥接 Session 已成功初始化！


26/04/01 10:11:44 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------+
|     namespace|
+--------------+
|bc2_cde_config|
|  bc2_cde_genl|
|  bc2_cde_rstk|
+--------------+

✅ 数据库列表拉取成功！


In [2]:
# 确保在正确的 Catalog 下

databases = [
    "local.bc2_cde_config",
    "local.bc2_cde_genl",
    "local.bc2_cde_rstk"
]

for db in databases:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")

In [3]:
# spark.sql("DROP DATABASE IF EXISTS local.c2_cde_rstk CASCADE");



In [3]:
spark.sql("SHOW DATABASES IN local").show()

+--------------+
|     namespace|
+--------------+
|bc2_cde_config|
|  bc2_cde_genl|
|  bc2_cde_rstk|
+--------------+



In [4]:
spark.sql("USE local")
spark.sql("CREATE DATABASE IF NOT EXISTS bc2_cde_rstk")
spark.sql("USE bc2_cde_rstk")

DataFrame[]

In [25]:
#007.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
# 执行建表 DDL
spark.sql("""CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF(
    RTN_RESP_EV_ID                          VARCHAR(100), 
    AI_ENT_CD                               VARCHAR(100), 
    RPT_POS_DT                              DATE, 
    SBMN_DTTM                               TIMESTAMP, 
    HALF_YR_END_DT                          DATE          ,
    APPR_MONY_BRKR_NM                       VARCHAR(2000) , 
    TNGBL_ASSET_AMT                         DECIMAL(30,10), 
    INV_AMT                                 DECIMAL(30,10), 
    OTH_NON_CUR_ASSET_NM                    VARCHAR(2000) , 
    OTH_NON_CUR_ASSET_AMT                   DECIMAL(30,10), 
    TOT_NON_CUR_ASSET_AMT                   DECIMAL(30,10), 
    CASH_AT_BANK_AND_IN_HAND_AMT            DECIMAL(30,10), 
    DEBTOR_AMT                              DECIMAL(30,10), 
    OTH_CUR_ASSET_NM                        VARCHAR(2000) , 
    OTH_CUR_ASSET_AMT                       DECIMAL(30,10), 
    TOT_CUR_ASSET_AMT                       DECIMAL(30,10), 
    TOT_ASSET_AMT                           DECIMAL(30,10), 
    CROR_AMT                                DECIMAL(30,10), 
    LN_AMT                                  DECIMAL(30,10), 
    DFR_TAX_AMT                             DECIMAL(30,10), 
    OTH_CUR_LIAB_NM                         VARCHAR(2000) , 
    OTH_CUR_LIAB_AMT                        DECIMAL(30,10), 
    TOT_CUR_LIAB_AMT                        DECIMAL(30,10), 
    LONG_TERM_LN_AMT                        DECIMAL(30,10), 
    OTH_LONG_TERM_LIAB_NM                   VARCHAR(2000) , 
    OTH_LONG_TERM_LIAB_AMT                  DECIMAL(30,10), 
    TOT_LONG_TERM_LIAB_AMT                  DECIMAL(30,10), 
    TOT_LIAB_AMT                            DECIMAL(30,10), 
    NET_ASSET_AMT                           DECIMAL(30,10), 
    PD_UP_SHR_CAP_AMT                       DECIMAL(30,10), 
    SHR_PREM_ACCT_AMT                       DECIMAL(30,10), 
    REVALQ_RESV_AMT                         DECIMAL(30,10), 
    OTH_RESV_NM                             VARCHAR(2000) , 
    OTH_RESV_AMT                            DECIMAL(30,10), 
    PRFT_AND_LOSS_ACCT_AMT                  DECIMAL(30,10), 
    TOT_SHRHLD_FUND_AMT                     DECIMAL(30,10), 
    ROW_VLD_STS_CD                          VARCHAR(255)  , 
    ROW_VLD_MSG_TXT                         VARCHAR(2000) ,
    TX_DT                                   DATE, 
    INSE_DTTM                               TIMESTAMP,       
    UPDT_DTTM                               TIMESTAMP       
) USING iceberg 
TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 已成功建立！")

表： AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 已成功建立！


In [26]:

#008.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
# 执行建表 DDL
spark.sql("""CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN(
    RTN_RESP_EV_ID                                                      VARCHAR(100),
    AI_ENT_CD                                                           VARCHAR(100),
    RPT_POS_DT                                                          DATE,
    SBMN_DTTM                                                           TIMESTAMP,
    HALF_YR_END_DT                                                      DATE,
    APPR_MONY_BRKR_NM                                                   VARCHAR(2000),
    TRAN_BSS_REVN_GEN_IND                                               VARCHAR(100),
    TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                 DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                           DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                                DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                          VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                         DECIMAL(30,10),
    FX_DRTV_FX_SWAP_AMT                                                 DECIMAL(30,10),
    FX_DRTV_NDF_AMT                                                     DECIMAL(30,10),
    FX_DRTV_NDO_AMT                                                     DECIMAL(30,10),
    FX_DRTV_VANILLA_FX_OPT_AMT                                          DECIMAL(30,10),
    FX_DRTV_OTH_FX_DRTV_NM                                              VARCHAR(2000),
    FX_DRTV_OTH_FX_DRTV_AMT                                             DECIMAL(30,10),
    INT_RT_DRTV_SNGL_CURY_IRS_AMT                                       DECIMAL(30,10),
    INT_RT_DRTV_CROSS_CURY_IRS_AMT                                      DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                    DECIMAL(30,10),
    INT_RT_DRTV_FRA_AMT                                                 DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_OPT_AMT                                          DECIMAL(30,10),
    INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                      VARCHAR(2000),
    INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                     DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_I_NM                                          VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_I_AMT                                         DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_II_NM                                         VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_II_AMT                                        DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_III_NM                                        VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_III_AMT                                       DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                           DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN        DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                                VARCHAR(2000),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                               DECIMAL(30,10),
    TOT_HK_RELVNT_BUSN_TRAN_AMT                                         DECIMAL(30,10),
    ROW_VLD_STS_CD                                                      VARCHAR(255),
    ROW_VLD_MSG_TXT                                                     VARCHAR(2000),
    TX_DT                                                               DATE,
    INSE_DTTM                                                           TIMESTAMP,
    UPDT_DTTM                                                           TIMESTAMP
) USING iceberg TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);


""")

print("表： AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 已成功建立！")

表： AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 已成功建立！


In [27]:

#009.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
# 执行建表 DDL
spark.sql("""
CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN(
    SURV_RESP_EV_ID                                                     VARCHAR(100),
    AI_ENT_CD                                                           VARCHAR(100),
    RPT_POS_DT                                                          DATE,
    SBMN_DTTM                                                           TIMESTAMP,
    HALF_YR_END_DT                                                      DATE,
    APPR_MONY_BRKR_NM                                                   VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                 DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                           DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                                DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                          VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                         DECIMAL(30,10),
    FX_DRTV_FX_SWAP_AMT                                                 DECIMAL(30,10),
    FX_DRTV_NDF_AMT                                                     DECIMAL(30,10),
    FX_DRTV_NDO_AMT                                                     DECIMAL(30,10),
    FX_DRTV_VANILLA_FX_OPT_AMT                                          DECIMAL(30,10),
    FX_DRTV_OTH_FX_DRTV_NM                                              VARCHAR(2000),
    FX_DRTV_OTH_FX_DRTV_AMT                                             DECIMAL(30,10),
    INT_RT_DRTV_SNGL_CURY_IRS_AMT                                       DECIMAL(30,10),
    INT_RT_DRTV_CROSS_CURY_IRS_AMT                                      DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                    DECIMAL(30,10),
    INT_RT_DRTV_FRA_AMT                                                 DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_OPT_AMT                                          DECIMAL(30,10),
    INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                      VARCHAR(2000),
    INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                     DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_I_NM                                          VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_I_AMT                                         DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_II_NM                                         VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_II_AMT                                        DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_III_NM                                        VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_III_AMT                                       DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                           DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN        DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                                VARCHAR(2000),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                               DECIMAL(30,10),
    TOT_HK_RELVNT_BUSN_TRAN_AMT                                         DECIMAL(30,10),
    ROW_VLD_STS_CD                                                      VARCHAR(255),
    ROW_VLD_MSG_TXT                                                     VARCHAR(2000),
    TX_DT                                                               DATE,
    INSE_DTTM                                                           TIMESTAMP,
    UPDT_DTTM                                                           TIMESTAMP
) USING iceberg TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN 已成功建立！")

表： AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN 已成功建立！


In [ ]:
#DROP TABLE [IF EXISTS] table_name;

# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")

# spark.sql("TRUNCATE TABLE local.bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF;")
# spark.sql("TRUNCATE TABLE local.bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN;")
# spark.sql("TRUNCATE TABLE local.bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN;")



DataFrame[]

In [5]:
#Check Table status before start ETL
print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF""").show()
print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN""").show()
print("AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN""").show()



AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF


26/04/01 10:13:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+---------+----------+---------+--------------+-----------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+--------------+---------------+-----+---------+---------+
|RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_CUR_LIAB_NM|OTH_CUR_LIAB_AMT|TOT_CUR_LIAB_AMT|LO

In [9]:
#Check parquets before start ETL
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251203_AMB_20251204000001_Passed_20251204.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20251203_AMB_20251204000001_Passed_20251204.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251203_AMB_20251204000001_Passed_20251204.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet`;""").show()




+--------------------+--------------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+-------------------+-----------------+---------+----------+--------------+
|      ROW_VLD_STS_CD|     ROW_VLD_MSG_TXT|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_CUR_LIAB_NM|OTH_CUR_LIAB_AMT|TOT_CUR_LIAB_AMT|LONG_TERM_LN_AMT|OTH_LONG_TERM_LIAB_NM|OTH_LONG_TERM_LIAB_A

s3a://bc2-raw-restricted-ide/processed/amb/CHC101/20251203/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet
s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet
s3a://bc2-raw-restricted-{{env}}/processed/amb/{{token_1}}/{{tx_dt}}/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_{{original_file_name}}.parquet

CREATE OR REPLACE TEMPORARY VIEW temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn
USING PARQUET OPTIONS (path 's3a://bc2-raw-restricted-{{env}}/processed/amb/{{token_1}}/{{tx_dt}}/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_{{original_file_name}}.parquet');


In [51]:
# 1. 定义 Python 变量
env = "ide"
token_1 = "CHC101"
#tx_dt = "20251203"
#original_file_name = "CHC101_20251203_AMB_20251203000001_Passed_20251203"


In [52]:
import ipywidgets as widgets
from IPython.display import display
import re

# 1. 定义新的文件名列表（不带后缀）
original_file_names = [
    "CHC101_20251203_AMB_20251203000001_Passed_20251203",
    "CHC101_20251203_AMB_20251204000001_Passed_20251204",
    "CHC101_20251205_AMB_20251205000001_Passed_20251205"
]

# --- 核心修改：提取函数 ---
def extract_tx_dt(name):
    # r'(\d{8})$' 表示匹配字符串末尾($)的连续8位数字(\d{8})
    match = re.search(r'(\d{8})$', str(name))
    if match:
        return match.group(1)
    return "Unknown"

# 2. 初始化赋值
original_file_name = original_file_names[0]
tx_dt = extract_tx_dt(original_file_name)

# 3. 创建下拉菜单
dropdown = widgets.Dropdown(
    options=original_file_names,
    value=original_file_name,
    description='选择文件:',
    style={'description_width': 'initial'},
    layout={'width': 'max-content'}
)

# 4. 回调函数
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        global original_file_name, tx_dt
        original_file_name = change['new']
        tx_dt = extract_tx_dt(original_file_name)
        # 这里用 print 确认变量确实变了
        print(f"✅ 变量已同步: original_file_name='{original_file_name}', tx_dt='{tx_dt}'")

dropdown.observe(on_change)
display(dropdown)

Dropdown(description='选择文件:', layout=Layout(width='max-content'), options=('CHC101_20251203_AMB_20251203000001…

In [79]:
# 直接打印查看当前变量的值
print(f"original_file_name: {original_file_name}")
print(f"tx_dt: {tx_dt}")
print(f'path: s3a://bc2-raw-restricted-{env}/processed/amb/{token_1}/{tx_dt}/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_{original_file_name}.parquet')

original_file_name: CHC101_20251205_AMB_20251205000001_Passed_20251205
tx_dt: 20251205
path: s3a://bc2-raw-restricted-ide/processed/amb/CHC101/20251205/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet


In [80]:

# 2. 构建 SQL 语句
sql_query_crtbl = f"""
CREATE OR REPLACE TEMPORARY VIEW temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn
USING PARQUET OPTIONS (
  path 's3a://bc2-raw-restricted-{env}/processed/amb/{token_1}/{tx_dt}/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_{original_file_name}.parquet'
);
"""


In [81]:

# 3. 执行 SQL 语句
spark.sql(sql_query_crtbl)

DataFrame[]

In [82]:
spark.sql("select * from temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn").show()

+--------------------+--------------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-------------------------------------+---------------------------+-------------------+-----------------+---------+

ETL:

In [83]:
#check data:
spark.sql("select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").show()

+--------------------+---------+----------+-------------------+--------------+-----------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-------------------------------------+-----------------------

In [84]:
#检查满足删除条件的数据
spark.sql("""SELECT * FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
""").show()

+---------------+---------+----------+---------+--------------+-----------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-------------------------------------+---------------------------+----------

In [85]:
#  构建 SQL 语句
sql_query_dl = f"""DELETE FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
"""

In [86]:

#  执行 SQL 语句
spark.sql(sql_query_dl)

DataFrame[]

In [87]:
#check data:
spark.sql("select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").show()

+--------------------+---------+----------+-------------------+--------------+-----------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-------------------------------------+-----------------------

In [88]:
#检查满足插入条件的数据
spark.sql("""SELECT * FROM temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
""").show()

+--------------------+--------------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-------------------------------------+---------------------------+-------------------+-----------------+---------+

In [89]:
#  构建 SQL 语句
sql_query_insrt = f"""INSERT INTO bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
    (    
        SURV_RESP_EV_ID                                                    ,
        AI_ENT_CD                                                          ,
        RPT_POS_DT                                                         ,
        SBMN_DTTM                                                          ,
        HALF_YR_END_DT                                                     ,
        APPR_MONY_BRKR_NM                                                  ,
        TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                ,
        TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                          ,
        TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                               ,
        TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                         ,
        TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                        ,
        FX_DRTV_FX_SWAP_AMT                                                ,
        FX_DRTV_NDF_AMT                                                    ,
        FX_DRTV_NDO_AMT                                                    ,
        FX_DRTV_VANILLA_FX_OPT_AMT                                         ,
        FX_DRTV_OTH_FX_DRTV_NM                                             ,
        FX_DRTV_OTH_FX_DRTV_AMT                                            ,
        INT_RT_DRTV_SNGL_CURY_IRS_AMT                                      ,
        INT_RT_DRTV_CROSS_CURY_IRS_AMT                                     ,
        INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                   ,
        INT_RT_DRTV_FRA_AMT                                                ,
        INT_RT_DRTV_INT_RT_OPT_AMT                                         ,
        INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                     ,
        INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                    ,
        OTH_MONY_MKT_OTC_DRTV_I_NM                                         ,
        OTH_MONY_MKT_OTC_DRTV_I_AMT                                        ,
        OTH_MONY_MKT_OTC_DRTV_II_NM                                        ,
        OTH_MONY_MKT_OTC_DRTV_II_AMT                                       ,
        OTH_MONY_MKT_OTC_DRTV_III_NM                                       ,
        OTH_MONY_MKT_OTC_DRTV_III_AMT                                      ,
        BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                          ,
        BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN       ,
        BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                               ,
        BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                              ,
        TOT_HK_RELVNT_BUSN_TRAN_AMT                                        ,
        ROW_VLD_STS_CD                                                     ,
        ROW_VLD_MSG_TXT                                                    ,
        TX_DT                                                              ,
        INSE_DTTM                                                          ,
        UPDT_DTTM
    )
SELECT
    sha2(CONCAT(
        COALESCE(AI_ENT_CD, ''),
        COALESCE(CAST(TO_DATE(RPT_POS_DT, 'yyyyMMdd') AS STRING), ''),
        COALESCE(CAST(TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss') AS STRING), '')
    ),256)                                                                                                                      AS SURV_RESP_EV_ID,
    CAST(AI_ENT_CD AS STRING)                                                                                                   AS AI_ENT_CD,
    TO_DATE(RPT_POS_DT, 'yyyyMMdd')                                                                                             AS RPT_POS_DT,
    TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss')                                                                                   AS SBMN_DTTM,
    CAST(SUBSTRING(TRIM(HALF_YR_END_DT), 1, 10) AS DATE)                                                                        AS HALF_YR_END_DT,
    CAST(APPR_MONY_BRKR_NM AS STRING)                                                                                           AS APPR_MONY_BRKR_NM,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                               AS TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                         AS TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                              AS TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT,
    CAST(TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM AS STRING)                                                                  AS TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                       AS TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT,      
    CAST(REPLACE(CAST(FX_DRTV_FX_SWAP_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                               AS FX_DRTV_FX_SWAP_AMT,
    CAST(REPLACE(CAST(FX_DRTV_NDF_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                                   AS FX_DRTV_NDF_AMT,
    CAST(REPLACE(CAST(FX_DRTV_NDO_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                                   AS FX_DRTV_NDO_AMT,
    CAST(REPLACE(CAST(FX_DRTV_VANILLA_FX_OPT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                        AS FX_DRTV_VANILLA_FX_OPT_AMT,
    CAST(FX_DRTV_OTH_FX_DRTV_NM AS STRING)                                                                                      AS FX_DRTV_OTH_FX_DRTV_NM,       
    CAST(REPLACE(CAST(FX_DRTV_OTH_FX_DRTV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                           AS FX_DRTV_OTH_FX_DRTV_AMT,   
    CAST(REPLACE(CAST(INT_RT_DRTV_SNGL_CURY_IRS_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                     AS INT_RT_DRTV_SNGL_CURY_IRS_AMT,   
    CAST(REPLACE(CAST(INT_RT_DRTV_CROSS_CURY_IRS_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                    AS INT_RT_DRTV_CROSS_CURY_IRS_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                  AS INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_FRA_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                               AS INT_RT_DRTV_FRA_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_INT_RT_OPT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                        AS INT_RT_DRTV_INT_RT_OPT_AMT,
    CAST(INT_RT_DRTV_OTH_INT_RT_DRTV_NM AS STRING)                                                                              AS INT_RT_DRTV_OTH_INT_RT_DRTV_NM,
    CAST(REPLACE(CAST(INT_RT_DRTV_OTH_INT_RT_DRTV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                   AS INT_RT_DRTV_OTH_INT_RT_DRTV_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_I_NM AS STRING)                                                                                  AS OTH_MONY_MKT_OTC_DRTV_I_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_I_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                       AS OTH_MONY_MKT_OTC_DRTV_I_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_II_NM AS STRING)                                                                                 AS OTH_MONY_MKT_OTC_DRTV_II_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_II_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                      AS OTH_MONY_MKT_OTC_DRTV_II_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_III_NM AS STRING)                                                                                AS OTH_MONY_MKT_OTC_DRTV_III_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_III_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                     AS OTH_MONY_MKT_OTC_DRTV_III_AMT,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                         AS BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN AS STRING), ',' ,'') AS DECIMAL(30,10))      AS BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN,
    CAST(BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM AS STRING)                                                                        AS BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                             AS BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT,
    CAST(REPLACE(CAST(TOT_HK_RELVNT_BUSN_TRAN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                       AS TOT_HK_RELVNT_BUSN_TRAN_AMT,
    CAST(ROW_VLD_STS_CD AS STRING)                                                                                              AS ROW_VLD_STS_CD,   
    CAST(ROW_VLD_MSG_TXT AS STRING)                                                                                             AS ROW_VLD_MSG_TXT,  
    TO_DATE(CAST('{{tx_dt}}' AS STRING), 'yyyyMMdd')                                                                            AS TX_DT,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                                                                   AS INSE_DTTM,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                                                                   AS UPDT_DTTM
FROM temp_view_amb_part_iii_tran_amt_of_hk_relvnt_busn t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
"""

In [90]:

#  执行 SQL 语句
spark.sql(sql_query_insrt)

DataFrame[]

In [4]:
#check data:
spark.sql("select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").show()

26/03/28 20:21:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+---------+----------+-------------------+--------------+-----------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-------------------------------------+-----------------------

In [92]:
# 方式 A：运行 SQL
# spark.sql("TRUNCATE TABLE bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")

# 方式 B：如果你想通过覆盖空 DataFrame 的方式（不推荐，TRUNCATE 更专业）
# spark.table("bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").limit(0).write.mode("overwrite").saveAsTable("...")

In [ ]:

# # 1. 读取表数据到 DataFrame
# df = spark.table("bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")

# # 2. 转换为 Pandas 并保存（路径建议选在你的 ldp 项目下）
# output_path = "/home/phil/ldp/csv_output/AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN.csv"
# df.toPandas().to_csv(output_path, index=False, encoding='utf-8-sig')

# print(f"✅ 文件已导出至: {output_path}")

✅ 文件已导出至: /home/phil/ldp/csv_output/AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN.csv
